# Monte Carlo option pricing: CPU vs GPU benchmark

Builds the repo, runs the CPU correctness test, then benchmarks four implementations of the same simulation (European + arithmetic Asian call, 252-step GBM paths):

| Implementation | What it is |
|---|---|
| CPU scalar 1T | one thread, one path at a time (the original reference) |
| CPU SIMD 1T | one thread, 32 paths stepped together so the loop vectorizes |
| CPU SIMD+OpenMP | the SIMD version on every core the runtime exposes |
| GPU | CUDA kernel, one path per thread, block-level reduction |

GPU speedups are reported against both the scalar 1T and the SIMD+OpenMP baselines.

**Setup:** Runtime → Change runtime type → **T4 GPU** → Save, then Runtime → Run all (about 5 minutes).

In [ ]:
# Hardware this run is on (also recorded in results/results.csv)
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!nvcc --version | tail -2
!lscpu | grep -E 'Model name|^CPU\(s\)|Thread|Core|Flags' | cut -c1-120
!echo "nproc: $(nproc)"

In [ ]:
REPO = 'https://github.com/LasyaRamachandruni/cuda-mc-option-pricing.git'
!rm -rf repo && git clone -q $REPO repo
%cd repo
!git log --oneline -1

## Build
`make mc` builds the CUDA benchmark (nvcc) plus the SIMD CPU baseline (g++ `-O3 -march=native -ffast-math -fopenmp`). `make test_cpu` builds the CPU correctness test.

In [ ]:
!make mc test_cpu

## Correctness test (CPU)
European call vs Black-Scholes (within 3 SE), scalar vs SIMD on the same paths, and the arithmetic Asian vs an independent control-variate reference. The same test runs in GitHub Actions.

In [ ]:
!./test_cpu

## Benchmark
65,536 → 4,194,304 paths. Each timing is the fastest of at least 3 runs (`--reps 3`), because Colab's vCPUs are shared. The scalar single-thread runs dominate the wall time.
The program exits non-zero if the CPU and GPU prices disagree by more than 0.1 SE (they simulate identical paths).

In [ ]:
!mkdir -p results
!./mc --reps 3 --label colab --csv results/results.csv

## Results table and chart
Writes `results/results.md` (markdown table) and `results/results.png`.

In [ ]:
!python3 scripts/report.py results/results.csv
from IPython.display import Image, display
display(Image('results/results.png'))

## Optional: profiling
Nsight Systems (timeline + kernel summary) and Nsight Compute (one `mc_kernel` launch). Either may be missing on the runtime, and `ncu` needs access to GPU performance counters; the cells print a message instead of failing.
`--max-scalar 0` skips the CPU baselines so the profile is just the GPU work.

In [ ]:
!if which nsys >/dev/null; then nsys profile --stats=true --force-overwrite true -o results/mc_nsys ./mc --max-paths 1048576 --max-scalar 0 --csv /tmp/nsys.csv 2>&1 | grep -A12 -E 'cuda_gpu_kern_sum|cuda_api_sum'; else echo 'nsys not installed on this runtime'; fi

In [ ]:
# --launch-skip 1 skips the warm-up launch; this profiles the first 1M-path launch.
!if which ncu >/dev/null; then ncu --kernel-name mc_kernel --launch-skip 1 --launch-count 1 --section SpeedOfLight --section Occupancy --section LaunchStats ./mc --min-paths 1048576 --max-paths 1048576 --max-scalar 0 --csv /tmp/ncu.csv; else echo 'ncu not installed on this runtime'; fi

## Save the results
Downloads `results/` (CSV, markdown table, chart) to add to the repo.

In [ ]:
!cd results && zip -q ../colab_results.zip results.csv results.md results.png
from google.colab import files
files.download('colab_results.zip')